# Data preprocessing pipeline - Ames Housing

Notebook này thực hiện: xử lý missing values, categorical encoding, scaling numerical features, train/validation split và đóng gói preprocessing pipeline.

Nguồn quyết định: `01_eda.ipynb`, `memory-bank/eda-findings.md`, `systemPatterns.md`, `project-management.md`. Repo hiện không có file PDF kiến trúc, nên kiến trúc được đối chiếu theo các markdown này.

Nguyên tắc: semantic missing xử lý trước; imputer, encoder và scaler chỉ được fit trên training split để tránh data leakage.

## 1. EDA -> quyết định preprocessing

EDA cho thấy:

- Dataset có 1.460 dòng train, 79 explanatory features; `Id` là identifier nên loại khỏi model.
- Raw có 6.965 token `NA`, nhưng nhiều token có nghĩa là structural absence: không có basement, garage, fireplace hoặc pool.
- `MasVnrType` có 864 category literal `None` hợp lệ và 8 `NA`; không được đọc bằng default NA rồi gộp hai ý nghĩa.
- Unknown missing còn tập trung ở `LotFrontage`, `MasVnrType`, `MasVnrArea`, `Electrical`, `BsmtExposure`, `BsmtFinType2`.
- `MSSubClass` và `MoSold` là categorical code dù dtype là số.
- Có 54 category levels xuất hiện dưới 5 lần; pipeline thử gom rare level bằng `min_frequency=5`.
- Test có level chỉ xuất hiện ở test, nổi bật là `MSSubClass=150`; encoder phải chịu được unseen category.
- `SalePrice` skewness khoảng 1.883; train bằng `log1p(SalePrice)`, dự đoán quay lại bằng `expm1`.
- EDA chưa kết luận xóa outlier `GrLivArea > 4000`, vì vậy notebook giữ nguyên để validation quyết định.

In [1]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

candidates = [
    Path.cwd(),
    Path.cwd() / "Tuan02" / "Lab02",
    Path.cwd().parent,
]
LAB_ROOT = next(
    (p.resolve() for p in candidates if (p / "data" / "train.csv").exists()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError("Không tìm thấy data/train.csv.")

DATA_DIR = LAB_ROOT / "data"
OUTPUT_DIR = LAB_ROOT / "outputs" / "preprocessing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("LAB_ROOT:", LAB_ROOT)
print("pandas:", pd.__version__)

LAB_ROOT: D:\HOCTAP\repo\Deep_Learning_Group\Tuan02\Lab02
pandas: 3.0.6


## 2. Đọc raw data và bảo toàn token `NA`

EDA yêu cầu `keep_default_na=False`. Nếu dùng mặc định của pandas, `NA` và category literal `None` có thể bị chuyển thành `NaN`, làm sai missing audit.

In [2]:
raw_train = pd.read_csv(DATA_DIR / "train.csv", keep_default_na=False)
raw_test = pd.read_csv(DATA_DIR / "test.csv", keep_default_na=False)
inferred_train = pd.read_csv(DATA_DIR / "train.csv")
inferred_test = pd.read_csv(DATA_DIR / "test.csv")

assert raw_train.shape == (1460, 81)
assert raw_test.shape == (1459, 80)
assert set(raw_train.columns) - set(raw_test.columns) == {"SalePrice"}
assert raw_train["Id"].is_unique and raw_test["Id"].is_unique

print("Train shape:", raw_train.shape)
print("Test shape:", raw_test.shape)
print("MasVnrType raw tokens:", raw_train["MasVnrType"].value_counts().head().to_dict())

Train shape: (1460, 81)
Test shape: (1459, 80)
MasVnrType raw tokens: {'None': 864, 'BrkFace': 445, 'Stone': 128, 'BrkCmn': 15, 'NA': 8}


## 3. Semantic missing-value preparation

Các rule được copy theo EDA:

- Basement absent khi `TotalBsmtSF == 0`.
- Garage absent khi `GarageCars == 0` và `GarageArea == 0`.
- Fireplace absent khi `Fireplaces == 0`.
- Pool absent khi `PoolArea == 0`.
- `Alley`, `Fence`, `MiscFeature` dùng `NA` trực tiếp cho trạng thái không có.
- Structural absence categorical đổi thành category `None`; `GarageYrBlt` giữ numeric `NaN` và có missing indicator.
- `NA` nhưng feature đi kèm cho thấy thành phần tồn tại sẽ giữ là unknown missing.

In [3]:
BASEMENT_COLUMNS = {
    "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2",
}
GARAGE_COLUMNS = {
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
}
UNCONDITIONAL_ABSENCE_COLUMNS = {"Alley", "Fence", "MiscFeature"}
STRUCTURAL_ABSENCE_COLUMNS = (
    BASEMENT_COLUMNS
    | GARAGE_COLUMNS
    | UNCONDITIONAL_ABSENCE_COLUMNS
    | {"FireplaceQu", "PoolQC"}
)
NUMERIC_STRUCTURAL_ABSENCE_COLUMNS = {"GarageYrBlt"}

def build_structural_absence_masks(raw):
    numeric = lambda col: pd.to_numeric(raw[col], errors="coerce")
    basement_absent = numeric("TotalBsmtSF").eq(0)
    garage_absent = numeric("GarageCars").eq(0) & numeric("GarageArea").eq(0)
    fireplace_absent = numeric("Fireplaces").eq(0)
    pool_absent = numeric("PoolArea").eq(0)

    masks = {}
    for column in STRUCTURAL_ABSENCE_COLUMNS | NUMERIC_STRUCTURAL_ABSENCE_COLUMNS:
        raw_na = raw[column].eq("NA")
        if column in BASEMENT_COLUMNS:
            masks[column] = raw_na & basement_absent
        elif column in GARAGE_COLUMNS or column == "GarageYrBlt":
            masks[column] = raw_na & garage_absent
        elif column == "FireplaceQu":
            masks[column] = raw_na & fireplace_absent
        elif column == "PoolQC":
            masks[column] = raw_na & pool_absent
        else:
            masks[column] = raw_na
    return pd.DataFrame(masks, index=raw.index).astype(bool)

def prepare_semantic_frame(raw, inferred):
    frame = raw.copy()
    absence_masks = build_structural_absence_masks(raw)

    for column in frame.columns:
        raw_na = raw[column].eq("NA")
        if column in STRUCTURAL_ABSENCE_COLUMNS:
            frame.loc[absence_masks[column], column] = "None"
            frame.loc[raw_na & ~absence_masks[column], column] = np.nan
        else:
            frame.loc[raw_na, column] = np.nan

    numeric_columns = inferred.select_dtypes(include=np.number).columns
    for column in numeric_columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame, absence_masks

train, train_absence_masks = prepare_semantic_frame(raw_train, inferred_train)
test, test_absence_masks = prepare_semantic_frame(raw_test, inferred_test)

# Các anomaly EDA đã phát hiện phải vẫn là unknown missing.
assert train.loc[train["Id"].eq(949), "BsmtExposure"].isna().all()
assert train.loc[train["Id"].eq(333), "BsmtFinType2"].isna().all()
assert test.loc[test["Id"].eq(2127), "GarageFinish"].isna().all()
assert test.loc[test["Id"].isin([2421, 2504, 2600]), "PoolQC"].isna().all()

print("Structural absence flags:", int(train_absence_masks.sum().sum()))
print("Unknown missing by column:")
print(train.isna().sum().loc[lambda s: s.gt(0)].sort_values(ascending=False))

Structural absence flags: 6687
Unknown missing by column:
LotFrontage     259
GarageYrBlt      81
MasVnrType        8
MasVnrArea        8
BsmtExposure      1
BsmtFinType2      1
Electrical        1
dtype: int64


## 4. Feature semantic types

Storage dtype không đồng nghĩa với modeling meaning. `MSSubClass` và `MoSold` phải đi qua categorical encoder. Ordinal features cũng one-hot ở baseline này vì `OrdinalEncoder` áp đặt khoảng cách tuyến tính giả giữa các mức chất lượng; nếu muốn dùng ordinal mapping thì phải tạo experiment riêng và so sánh validation.

In [4]:
ORDINAL_COLUMNS = {
    "OverallQual", "OverallCond", "ExterQual", "ExterCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "HeatingQC", "KitchenQual", "Functional", "FireplaceQu", "GarageFinish",
    "GarageQual", "GarageCond", "PoolQC", "Fence", "LotShape", "LandSlope",
    "PavedDrive",
}
CATEGORICAL_CODE_COLUMNS = {"MSSubClass", "MoSold"}
TEMPORAL_COLUMNS = {"YearBuilt", "YearRemodAdd", "GarageYrBlt", "YrSold"}

TARGET_COLUMN = "SalePrice"
ID_COLUMN = "Id"
feature_columns = [c for c in train.columns if c not in {TARGET_COLUMN, ID_COLUMN}]

categorical_features = [
    c for c in feature_columns
    if (
        c in ORDINAL_COLUMNS
        or c in CATEGORICAL_CODE_COLUMNS
        or not pd.api.types.is_numeric_dtype(train[c])
    )
]
numeric_features = [c for c in feature_columns if c not in categorical_features]

for column in categorical_features:
    train[column] = train[column].where(train[column].notna(), np.nan).astype(object)
    test[column] = test[column].where(test[column].notna(), np.nan).astype(object)

print("Numeric features:", len(numeric_features))
print("Categorical/ordinal features:", len(categorical_features))
print("Temporal features treated as numeric:",
      sorted(TEMPORAL_COLUMNS & set(numeric_features)))

Numeric features: 32
Categorical/ordinal features: 47
Temporal features treated as numeric: ['GarageYrBlt', 'YearBuilt', 'YearRemodAdd', 'YrSold']


## 5. Train/validation split và target transform

Split trước khi fit imputer, encoder và scaler. Validation chỉ được transform, không được fit. Đây là yêu cầu leakage check trong memory bank.

Do target lệch phải, model học `log1p(SalePrice)`. Metric nội bộ dùng log-RMSE, phù hợp định hướng của EDA và project architecture.

In [5]:
X = train[feature_columns].copy()
y = train[TARGET_COLUMN].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)
y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train log mean/std:", y_train_log.mean(), y_train_log.std())
print("y_valid log mean/std:", y_valid_log.mean(), y_valid_log.std())

X_train: (1168, 79)
X_valid: (292, 79)
y_train log mean/std: 12.030658310971573 0.3906060203170431
y_valid log mean/std: 11.997653730705732 0.43272654582762493


## 6. Xây dựng preprocessing pipeline

**Numeric branch:** unknown missing dùng median học từ train; `add_indicator=True` giữ tín hiệu missing; `StandardScaler` đưa các biến về scale phù hợp cho Ridge/MLP.

**Categorical branch:** structural absence `None` được giữ như category; missing thật thành `Unknown`. `min_frequency=5` là thử nghiệm có căn cứ vì EDA thấy 54 level hiếm. `handle_unknown='infrequent_if_exist'` xử lý level mới ở test như `MSSubClass=150`.

In [6]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=5,
        sparse_output=False,
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

preprocessing_model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=20.0)),
])

print(preprocessing_model_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['LotFrontage', 'LotArea',
                                                   'YearBuilt', 'YearRemodAdd',
                                                   'MasVnrArea', 'BsmtFinSF1',
                                                   'BsmtFinSF2', 'BsmtUnfSF',
                                                   'TotalBsmtSF', '1stFlrSF',
                                                   '2ndFlrSF', 'LowQ

## 7. Fit, đánh giá và kiểm tra output

Chỉ `X_train` được dùng trong `fit`. Sau khi fit, validation đi qua pipeline bằng `predict`. Ma trận output phải không còn missing/inf và số chiều được ghi lại để bàn giao cho MLP/DataLoader.

In [7]:
preprocessing_model_pipeline.fit(X_train, y_train_log)

valid_pred_log = preprocessing_model_pipeline.predict(X_valid)
valid_rmse_log = mean_squared_error(y_valid_log, valid_pred_log) ** 0.5
valid_mae_log = mean_absolute_error(y_valid_log, valid_pred_log)

transformed_train = preprocessing_model_pipeline.named_steps["preprocessor"].transform(X_train)
transformed_valid = preprocessing_model_pipeline.named_steps["preprocessor"].transform(X_valid)

print(f"Validation log-RMSE: {valid_rmse_log:.4f}")
print(f"Validation log-MAE:  {valid_mae_log:.4f}")
print("Transformed train shape:", transformed_train.shape)
print("Transformed validation shape:", transformed_valid.shape)
print("All transformed values finite:", np.isfinite(transformed_train).all())

Validation log-RMSE: 0.1437


Validation log-MAE:  0.0963
Transformed train shape: (1168, 314)
Transformed validation shape: (292, 314)
All transformed values finite: True


## 8. Ghi lại feature names và lưu pipeline

Lưu cả preprocessing và model trong cùng một object giúp inference dùng đúng thứ tự: semantic preparation -> imputation -> encoding -> scaling -> model. Không cần xử lý thủ công lại dữ liệu mới.

In [8]:
feature_names = preprocessing_model_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

feature_audit_after_preprocessing = pd.DataFrame({
    "feature_name": feature_names,
    "position": np.arange(len(feature_names)),
})
feature_audit_after_preprocessing.to_csv(
    OUTPUT_DIR / "transformed_feature_names.csv", index=False
)

PIPELINE_PATH = OUTPUT_DIR / "ames_preprocessing_ridge_pipeline.pkl"
joblib.dump(preprocessing_model_pipeline, PIPELINE_PATH)

valid_predictions = pd.DataFrame({
    "Id": train.loc[X_valid.index, ID_COLUMN].values,
    "actual_SalePrice": y_valid.values,
    "predicted_log_SalePrice": valid_pred_log,
    "predicted_SalePrice": np.expm1(valid_pred_log),
})
valid_predictions.to_csv(
    OUTPUT_DIR / "validation_predictions.csv", index=False
)

print("Output features:", len(feature_names))
print("Saved pipeline:", PIPELINE_PATH)
print("Saved feature audit and validation predictions.")

Output features: 314
Saved pipeline: D:\HOCTAP\repo\Deep_Learning_Group\Tuan02\Lab02\outputs\preprocessing\ames_preprocessing_ridge_pipeline.pkl
Saved feature audit and validation predictions.


## 9. Handoff cho modeling

Notebook đã hoàn thành:

1. Missing semantics theo row-level rules từ EDA.
2. Numeric imputation bằng median và missing indicators.
3. Categorical imputation bằng `Unknown`, giữ `None` structural absence.
4. One-hot encoding cho categorical, ordinal và numeric-code features.
5. Scaling numerical features bằng `StandardScaler`.
6. Train/validation split với seed 42.
7. Target transform `log1p`.
8. Pipeline chống leakage và xử lý unseen categories.
9. Lưu pipeline, feature names và validation predictions vào `outputs/preprocessing/`.

Experiment tiếp theo: so sánh `min_frequency=5` với encoder không gom rare levels; thử Ridge/ElasticNet/MLP; và đánh giá riêng kịch bản giữ hoặc loại hai outlier `GrLivArea > 4000`.